In [1]:
# Imports
import pickle

import jax
import jaxopt
import jax.nn as jnn
import jax.numpy as jnp
from functools import partial

import time as tt
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
plt.rcParams['font.size'] = 18

import os
import corner
import numpy as np
import multiprocessing as mp

import dynesty
import dynesty.utils as dyut

from potentials import *
from integrants import *
from utils import *

from potentials import NFW_acceleration
from densities import MiyamotoNagai_density

from CylindricalSpline import get_phi_m, get_acc

from main import *

import time as tt

In [2]:
#Hyperparameters for dynesty
ndim = 5
nlive = 500
PATH_DATA = f'/data/dc824-2/ScharMAX_first_tests'

with open('./IC_axisymmetric_disc.pkl', 'rb') as f:
    ic = pickle.load(f)
w0 = jnp.array([ic['x'], ic['y'], ic['z'], ic['vx'], ic['vy'], ic['vz']]).T

with open('./axisymmetric_disc.pkl', 'rb') as f:
    data = pickle.load(f)
w0_data = jnp.array([data['x'], data['y'], data['z'], data['vx'], data['vy'], data['vz']]).T
density_data = histogram3d(w0_data[:, :3])

r_data    = jnp.linalg.norm(w0_data[:, :2], axis=-1)
vr_data   = (w0_data[:, 0] * w0_data[:, 3] + w0_data[:, 1] * w0_data[:, 4]) / r_data
vr_data_binned  = histogram3d(w0_data[:, :3], vr_data) / density_data
vr2_data_binned  = histogram3d(w0_data[:, :3], vr_data**2) / density_data
sigmavr_data_binned = jnp.sqrt(vr2_data_binned - vr_data_binned**2)

vphi_data = -(-w0_data[:, 1] * w0_data[:, 3] + w0_data[:, 0] * w0_data[:, 4]) / r_data
vphi_data_binned  = histogram3d(w0_data[:, :3], vphi_data) / density_data
vphi2_data_binned  = histogram3d(w0_data[:, :3], vphi_data**2) / density_data
sigmavphi_data_binned = jnp.sqrt(vphi2_data_binned - vphi_data_binned**2)

vz_data = w0_data[:, 5]
vz_data_binned = histogram3d(w0_data[:, :3], w0_data[:, 5]) / density_data
vz2_data_binned = histogram3d(w0_data[:, :3], vz_data**2) / density_data
sigmavz_data_binned = jnp.sqrt(vz2_data_binned - vz_data_binned**2)

dict_data = {
    'w0': w0,
    'density_data': density_data,
    'vr_data_binned': vr_data_binned,
    'sigmavr_data_binned': sigmavr_data_binned,
    'vphi_data_binned': vphi_data_binned,
    'sigmavphi_data_binned': sigmavphi_data_binned,
    'vz_data_binned': vz_data_binned,
    'sigmavz_data_binned': sigmavz_data_binned
}

In [3]:
p = np.random.uniform(0.0, 1.0, size=(ndim,))
params = prior_transform(p)
params

Array([11.0519   , 22.481188 ,  8.731089 ,  4.8942704,  0.9039195],      dtype=float32)

In [4]:
params_halo_pot = {
    'logM': params[0],
    'Rs':params[1],
    'a':1.0,
    'b':1.0,
    'c':1.0,
    'x_origin':0.0,
    'y_origin':0.0,
    'z_origin':0.0,
    'dirx':0.0,
    'diry':0.0,
    'dirz':1.0
}

params_disk_rho = {
    'logM': params[2],
    'Rs': params[3],
    'Hs': params[4],
    'x_origin': 0.0,
    'y_origin': 0.0,
    'z_origin': 0.0,
    'dirx': 0.0,
    'diry': 0.0,
    'dirz': 1.0
}

In [7]:
NR, NZ, Rmin, Rmax, Zmin, Zmax, Mmax = 50, 30, 1e-2, 30.0, 1e-3, 15.0, 8.
Nphi = 200
N_int = 10_000

a = tt.time()
dict_phi = jax.block_until_ready(get_phi_m(MiyamotoNagai_density, params_disk_rho, NR, NZ, Rmin, Rmax, Zmin, Zmax, Mmax, Nphi, N_int))
b = tt.time()
print(f'Time to compute phi_m: {b - a} seconds')

Time to compute phi_m: 1.6549248695373535 seconds


In [9]:
@jax.jit
def acc_fn(x, y, z, params_halo_pot, dict_phi):
    a_halo = NFW_acceleration(x, y, z, params_halo_pot)
    a_disk = get_acc(x, y, z, dict_phi)
    return a_halo + a_disk

def _split(w):
    return w[:3], w[3:]

def _merge(r, v):
    return jnp.concatenate([r, v], axis=0)

@partial(jax.jit, static_argnames=('acc_fn', 'n_steps', 'unroll'))
def integrate_leapfrog_traj(w0, acc_fn, params_halo_pot, dict_phi, n_steps, dt = 0.010, t0 = 0.0, unroll=True):
    """Leapfrog (KDK) — returns final time and final state only."""

    def step(carry, _):
        t, y = carry
        r, v = _split(y)

        # params['t'] = t  # Update time-dependent parameters
        a0 = acc_fn(*r, params_halo_pot, dict_phi) # , params)
        v_half = v + 0.5 * dt * a0
        r_new = r + dt * v_half
        t_new = t + dt

        # params['t'] = t_new  # Update time-dependent parameters
        a1 = acc_fn(*r_new, params_halo_pot, dict_phi) # , params)
        v_new = v_half + 0.5 * dt * a1
        y_new = _merge(r_new, v_new)
        return (t_new, y_new), (t_new, y_new)

    (_, _), (tN, wN) = jax.lax.scan(step, (t0, w0), xs=None, length=n_steps, unroll=unroll)
    return tN, wN

time = 2. #Gyr
n_steps = 1000
dt = time / n_steps
unroll = False
initial_time = 0.0

a = tt.time()
time, xv = jax.block_until_ready(
                jax.vmap(integrate_leapfrog_traj, in_axes=(0, None, None, None, None, None, None, None))(dict_data['w0'], acc_fn, params_halo_pot, dict_phi, n_steps, dt, initial_time, unroll)
            )
b = tt.time()
print(f'Time to compute xv: {b - a} seconds')

Time to compute xv: 17.046624898910522 seconds


In [16]:
density_model = jax.block_until_ready(jax.vmap(histogram3d)(xv[:, :, :3]))

In [30]:
A = density_model.reshape(len(xv), -1).T.astype(jnp.float32)/n_steps
y = dict_data['density_data'].reshape(-1).astype(jnp.float32)
sig = jnp.sqrt(y + 1.0).astype(jnp.float32)  # Poisson noise + 1.0 floor

@jax.jit
def _nll_z(z, A, y, sig, l2):
    x = jnn.softplus(z)  # strictly positive
    r = (A @ x - y) / sig
    return 0.5 * jnp.dot(r, r) + 0.5 * l2 * jnp.dot(x, x)
_nll_z = jax.value_and_grad(_nll_z)

@jax.jit
def solve_lbfgs_softplus(A, y, sigma, l2=1e-3, maxiter=500, tol=1e-6):
    z0 = jnp.zeros(A.shape[1], A.dtype)
    solver = jaxopt.LBFGS(fun=_nll_z, value_and_grad=True, maxiter=maxiter, tol=tol)
    res = solver.run(z0, A, y, sigma, l2)
    x_hat = jnn.softplus(res.params)
    return x_hat

weights = jax.block_until_ready(solve_lbfgs_softplus(A, y, sig, l2=1e-3, maxiter=300))

In [28]:
weights_binned = histogram3d(xv[:, :, :3].reshape(-1, 3), jnp.repeat(weights, n_steps).ravel())

r_model    = jnp.linalg.norm(xv[:, :, :2], axis=-1)
vr_model   = (xv[:, :, 0] * xv[:, :, 3] + xv[:, :, 1] * xv[:, :, 4]) / r_model
vr_binned  = histogram3d(xv[:, :, :3].reshape(-1, 3), (vr_model*weights[:, None]).ravel())/(weights_binned+EPSILON)
vr2_binned  = histogram3d(xv[:, :, :3].reshape(-1, 3), (vr_model**2 * weights[:, None]).ravel())/(weights_binned+EPSILON)
sigmavr_binned = jnp.sqrt(jnp.clip(vr2_binned - vr_binned**2, a_min=0.0))

vphi = -(-xv[:, :, 1] * xv[:, :, 3] + xv[:, :, 0] * xv[:, :, 4]) / r_model
vphi_binned  = histogram3d(xv[:, :, :3].reshape(-1, 3), (vphi*weights[:, None]).ravel())/(weights_binned+EPSILON)
vphi2_binned  = histogram3d(xv[:, :, :3].reshape(-1, 3), (vphi**2 * weights[:, None]).ravel())/(weights_binned+EPSILON)
sigmavphi_binned = jnp.sqrt(jnp.clip(vphi2_binned - vphi_binned**2, a_min=0.0))

vz_binned = histogram3d(xv[:, :, :3].reshape(-1, 3), (xv[:, :, 5]*weights[:, None]).ravel())/(weights_binned+EPSILON)
vz2_binned = histogram3d(xv[:, :, :3].reshape(-1, 3), (xv[:, :, 5]**2 * weights[:, None]).ravel())/(weights_binned+EPSILON)
sigmavz_binned = jnp.sqrt(jnp.clip(vz2_binned - vz_binned**2, a_min=0.0))

In [158]:
from CylindricalSpline import evaluate_phi

In [ ]:
x, y, z = np.random.uniform(-15.0, 15.0, size=(3,10000000))
# jax.vmap(get_acc, in_axes=(0,0,0,None))(x,y,z,dict_phi)
jax.vmap(jax.grad(evaluate_phi), in_axes=(0,0,0,None))(x,y,z,dict_phi)

In [116]:
x, y, z = np.random.uniform(-15.0, 15.0, size=(3,10000000))
jax.vmap(NFW_acceleration, in_axes=(0,0,0,None))(x,y,z,params_halo_pot)

Array([[ 1024.1385 ,  -263.23358,  2012.188  ],
       [-1554.898  ,   904.7646 ,  2766.6726 ],
       [ -502.81058,  1363.1133 , -1606.0945 ],
       ...,
       [-2215.6995 ,   232.71416,  1689.2145 ],
       [-1294.511  , -1069.7363 ,  -920.12976],
       [-1988.6317 ,   395.8862 ,  1055.0232 ]], dtype=float32)